# **SUBSCRIBER**

In [ ]:
import json
import os
import csv
import random
import time
from paho.mqtt import client as mqtt_client

# TODO: UPDATE FOLDER PATH and OUTPUT FILE-NAME IN HERE
SAVE_FOLDER = "Data"
CSV_FILENAME = "plant/env/predicted"
OUTPUT_CSV = os.path.join(SAVE_FOLDER, CSV_FILENAME)

STATE = {
    "writer_initialized": False,
    "fieldnames": None,
    "first_message_received": False,
    "last_msg_time": None,
}

# NOTE: MQTT CONFIGURATION
MQTT_CONFIG = {
    "BROKER": "broker.emqx.io",
    "PORT": 1883,
    "TOPIC": "plant/env/predicted",
    "CLIENT_ID": f"ALPHA-I-{random.randint(0,100)}",
    "KEEPALIVE": 120,
}

def connect_mqtt():
    client = mqtt_client.Client(mqtt_client.CallbackAPIVersion.VERSION1, MQTT_CONFIG["CLIENT_ID"])
    client.on_connect = on_connect
    client.on_message = on_message
    client.connect(MQTT_CONFIG["BROKER"], MQTT_CONFIG["PORT"], MQTT_CONFIG["KEEPALIVE"])
    return client


def on_connect(client, userdata, flags, rc):
    if rc == 0:
        print("Connected to MQTT broker.")
        client.subscribe(MQTT_CONFIG["TOPIC"])
        print(f"Subscribed to topic: {MQTT_CONFIG['TOPIC']}")
    else:
        print(f"Failed to connect, return code {rc}")


def save_data_csv(data: dict):
    os.makedirs(SAVE_FOLDER, exist_ok=True)

    if not STATE["writer_initialized"]:
        STATE["fieldnames"] = list(data.keys())
        file_exists = os.path.isfile(OUTPUT_CSV)

        with open(OUTPUT_CSV, "a", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=STATE["fieldnames"])
            if not file_exists:
                writer.writeheader()
            writer.writerow(data)

        STATE["writer_initialized"] = True
        return

    with open(OUTPUT_CSV, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=STATE["fieldnames"])
        writer.writerow(data)


def on_message(client, userdata, msg):
    STATE["first_message_received"] = True
    STATE["last_msg_time"] = time.time()

    try:
        payload_str = msg.payload.decode("utf-8")
        data = json.loads(payload_str)
        save_data_csv(data)
        print(f"Saved row to {OUTPUT_CSV}: {data}")

    except Exception as e:
        print(f"Error processing message: {e}")


def subscriber(IDLE_TIMEOUT = 10):
    client = connect_mqtt()
    print("Starting MQTT Subscribe. (Timeout Activate)")

    client.loop_start()
    try:
        while True:
            if STATE["first_message_received"]:
                idle = time.time() - STATE["last_msg_time"]
                if idle >= IDLE_TIMEOUT:
                    print(
                        f"No Subscribe message for {IDLE_TIMEOUT}s -- stop subscribe process."
                    )
                    break
            time.sleep(0.5)

    except KeyboardInterrupt:
        print("\nStopping subscriber manually...")

    finally:
        client.loop_stop()
        client.disconnect()
        print("Disconnected from MQTT broker.")


if __name__ == "__main__":
    subscriber()


C:\Users\ASUS\AppData\Local\Temp\ipykernel_14452\3874357487.py:30: DeprecationWarning: Callback API version 1 is deprecated, update to latest version
  client = mqtt_client.Client(mqtt_client.CallbackAPIVersion.VERSION1, MQTT_CONFIG["CLIENT_ID"])


Starting MQTT Subscribe. (Timeout Activate)
Connected to MQTT broker.
Subscribed to topic: plant/env/predicted

Stopping subscriber manually...
Disconnected from MQTT broker.
